# P3 confirmatory labeling session (one cohort)

Set `COHORT` and `QISKIT` in the first code cell: `C261` / `2.6.1` or `C252` / `2.5.2`. Settings: **CPU**, **Internet OFF**, *Pin to original environment*. Inputs: `p3-code` (the registered version) and `p3-wheelhouse-qiskit-<version>`. Run with **Save & Run All (Commit)**. Expected time about 3 to 5 hours. Resumable: if it stops, attach this notebook's previous output as an input and run again.

Run only after the registration has been pushed.

In [ ]:
COHORT = "C261"      # C261 (Qiskit 2.6.1) or C252 (Qiskit 2.5.2)
QISKIT = "2.6.1"
import glob, os
def find_one(pattern):
    hits = sorted(glob.glob(pattern, recursive=True))
    assert hits, f"not found: {pattern}  (is the dataset attached?)"
    return hits[0]
CODE = find_one("/kaggle/input/**/cart/src/cart/__init__.py").split("/cart/src/cart/__init__.py")[0]   # .../p3-code
print("code dataset:", CODE)
import json, platform, shutil, subprocess, sys, time
WH = os.path.dirname(find_one("/kaggle/input/**/wheelhouse/MANIFEST.json"))
assert json.load(open(os.path.join(WH, "MANIFEST.json")))["qiskit_requested"] == QISKIT, "wrong wheelhouse attached"
print("wheelhouse:", WH, json.load(open(os.path.join(WH, "MANIFEST.json")))["import_check"])
for k, v in {"RAYON_NUM_THREADS": "1", "OMP_NUM_THREADS": "1", "QISKIT_IN_PARALLEL": "TRUE",
             "OPENBLAS_NUM_THREADS": "1", "MKL_NUM_THREADS": "1"}.items():
    os.environ[k] = v
WORK = "/kaggle/working/p3"
if not os.path.exists(WORK):
    shutil.copytree(CODE, WORK)
SITE = "/kaggle/working/site"      # Kaggle's Python has no ensurepip, so no venv: an offline --target install
if not os.path.exists(os.path.join(SITE, "qiskit")):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-index", "--find-links", WH,
                    "--target", SITE, "-r", os.path.join(WH, "requirements_p3.txt")], check=True)
PY = sys.executable
PYPATH = f"{SITE}:{WORK}/cart/src:{WORK}/obsaware/src"   # our packages come first, ahead of Kaggle's preinstalled ones
print(subprocess.run([PY, "-c", "import qiskit, qiskit_aer, sys; print(sys.version.split()[0], qiskit.__version__, qiskit.__file__, qiskit_aer.__version__)"],
                     capture_output=True, text=True, env=dict(os.environ, PYTHONPATH=PYPATH)).stdout)


In [ ]:
# Pin this kernel (and every child process it starts) to one core.
TASKSET = shutil.which("taskset")
if TASKSET:
    print(subprocess.run([TASKSET, "-cp", "0", str(os.getpid())], capture_output=True, text=True).stdout)
else:
    print("WARNING: taskset not available; running unpinned (recorded in the run log)")


In [ ]:
# Resume support: if a previous K1 output is attached as input, start from it.
OUT = f"/kaggle/working/out_{COHORT}"
prev = sorted(glob.glob(f"/kaggle/input/**/out_{COHORT}/units", recursive=True))
if prev and not os.path.exists(OUT):
    shutil.copytree(os.path.dirname(prev[0]), OUT)
    print("resuming from", prev[0])
env = dict(os.environ, PYTHONPATH=PYPATH)
cmd = ([TASKSET, "-c", "0"] if TASKSET else []) + [PY, "-u", f"{WORK}/obsaware/scripts/label_cohort.py", "--out", OUT, "--cohort", COHORT]
if os.environ.get("P3_MAX_WIDTH"):          # local dry runs only; never set on Kaggle
    cmd += ["--max-width", os.environ["P3_MAX_WIDTH"]]
if os.environ.get("P3_ONLY_EVENTS"):        # local dry runs only; never set on Kaggle
    cmd += ["--only-events", os.environ["P3_ONLY_EVENTS"]]
p = subprocess.Popen(cmd, cwd=f"{WORK}/cart", env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in p.stdout:
    print(line, end="")
print("exit code", p.wait())


In [ ]:
shutil.make_archive(f"/kaggle/working/out_{COHORT}", "zip", OUT)
print(os.listdir("/kaggle/working"))
